# Выявление мошеннических финансовых транзакций

Полный ML-цикл для несбалансированной бинарной классификации: подготовка
признаков, разведочный анализ, кросс-валидация без утечки данных, подбор
гиперпараметров через Optuna и сравнение отдельных моделей с ансамблями.

Исследование сопровождает [статью](../paper/article_clean.pdf) Д. А. Антипенко,
М. Е. Суханова и В. Ю. Радыгина (2026).

> **Данные.** Оригинальный `FraudShield_Banking_Data.csv` не включён в
> репозиторий и сейчас недоступен публично. Инструкция и схема находятся в
> [`data/README.md`](../data/README.md). Ноутбук не подменяет источник другим
> датасетом и не хранит неподтверждённые метрики.


## 1. Окружение

Число Optuna-испытаний можно изменить переменной `FRAUD_N_TRIALS`.
Исходное исследование использовало 40; более быстрый портфолио-запуск по
умолчанию использует 15.


In [ ]:
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:  # позволяет выполнять код и вне Jupyter
    display = print
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42
N_TRIALS = int(os.getenv("FRAUD_N_TRIALS", "15"))
N_JOBS = int(os.getenv("FRAUD_N_JOBS", "-1"))
print(f"Optuna trials per model: {N_TRIALS}; parallel jobs: {N_JOBS}")


## 2. Загрузка и проверка исходного CSV


In [ ]:
expected_columns = {
    "Transaction_ID", "Customer_ID", "Merchant_ID", "Device_ID", "IP_Address",
    "Transaction_Date", "Transaction_Time", "Transaction_Location",
    "Customer_Home_Location", "Transaction_Type", "Merchant_Category", "Card_Type",
    "Is_International_Transaction", "Is_New_Merchant", "Unusual_Time_Transaction",
    "Transaction_Amount (in Million)", "Distance_From_Home",
    "Account_Balance (in Million)", "Daily_Transaction_Count",
    "Weekly_Transaction_Count", "Avg_Transaction_Amount (in Million)",
    "Max_Transaction_Last_24h (in Million)", "Failed_Transaction_Count",
    "Previous_Fraud_Count", "Fraud_Label",
}

env_path = os.getenv("FRAUD_DATA_PATH")
candidates = [
    Path(env_path).expanduser() if env_path else None,
    Path("data/FraudShield_Banking_Data.csv"),
    Path("../data/FraudShield_Banking_Data.csv"),
]
data_path = next((path.resolve() for path in candidates if path and path.is_file()), None)

if data_path is None:
    raise FileNotFoundError(
        "Не найден FraudShield_Banking_Data.csv. Поместите его в data/ или "
        "задайте переменную FRAUD_DATA_PATH. См. data/README.md."
    )

df = pd.read_csv(data_path)
missing_columns = sorted(expected_columns - set(df.columns))
if missing_columns:
    raise ValueError("В CSV отсутствуют столбцы: " + ", ".join(missing_columns))

unexpected_labels = sorted(
    set(df["Fraud_Label"].dropna().astype(str).unique()) - {"Normal", "Fraud"}
)
if unexpected_labels:
    raise ValueError(f"Неожиданные значения Fraud_Label: {unexpected_labels}")

print(f"Источник: {data_path}")
print(f"Размер исходных данных: {df.shape}")
display(df.head())


## 3. Подготовка и конструирование признаков

Формируются временные признаки, отношения суммы операции к типичному
уровню и балансу, индикатор несовпадения географии и несколько комбинаций
факторов риска.


In [ ]:
df = df.copy()
df["Fraud_Label"] = df["Fraud_Label"].map({"Normal": 0, "Fraud": 1})
df["Transaction_DateTime"] = pd.to_datetime(
    df["Transaction_Date"].astype(str) + " " + df["Transaction_Time"].astype(str),
    errors="coerce",
)

df["hour"] = df["Transaction_DateTime"].dt.hour
df["day_of_week"] = df["Transaction_DateTime"].dt.dayofweek
df["is_night"] = ((df["hour"] >= 0) & (df["hour"] < 6)).astype(int)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

eps = 1e-9
df["amount_to_avg_ratio"] = (
    df["Transaction_Amount (in Million)"]
    / (df["Avg_Transaction_Amount (in Million)"].abs() + eps)
)
df["amount_to_balance_ratio"] = (
    df["Transaction_Amount (in Million)"]
    / (df["Account_Balance (in Million)"].abs() + eps)
)
df["is_location_mismatch"] = (
    df["Transaction_Location"] != df["Customer_Home_Location"]
).astype(int)
df["intl_and_unusual"] = (
    (df["Is_International_Transaction"] == "Yes")
    & (df["Unusual_Time_Transaction"] == "Yes")
).astype(int)
df["intl_or_unusual"] = (
    (df["Is_International_Transaction"] == "Yes")
    | (df["Unusual_Time_Transaction"] == "Yes")
).astype(int)
df["new_merchant_and_intl"] = (
    (df["Is_New_Merchant"] == "Yes")
    & (df["Is_International_Transaction"] == "Yes")
).astype(int)
df["risk_score"] = (
    (df["Is_International_Transaction"] == "Yes").astype(int)
    + (df["Unusual_Time_Transaction"] == "Yes").astype(int)
    + (df["Is_New_Merchant"] == "Yes").astype(int)
    + df["is_location_mismatch"]
)

categorical_features = [
    "Transaction_Type", "Merchant_Category", "Card_Type",
    "Is_International_Transaction", "Is_New_Merchant", "Unusual_Time_Transaction",
]
numerical_features = [
    "Transaction_Amount (in Million)", "Distance_From_Home",
    "Account_Balance (in Million)", "Daily_Transaction_Count",
    "Weekly_Transaction_Count", "Avg_Transaction_Amount (in Million)",
    "Max_Transaction_Last_24h (in Million)", "Failed_Transaction_Count",
    "Previous_Fraud_Count", "hour", "day_of_week", "is_night", "is_weekend",
    "amount_to_avg_ratio", "amount_to_balance_ratio", "is_location_mismatch",
    "intl_and_unusual", "intl_or_unusual", "new_merchant_and_intl", "risk_score",
]

df_clean = df.dropna(subset=["Fraud_Label"]).copy()
X = df_clean[numerical_features + categorical_features].copy()
y = df_clean["Fraud_Label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Удалено строк без целевой метки: {len(df) - len(df_clean)}")
print(f"Train: {X_train.shape}; test: {X_test.shape}")
print(f"Fraud rate — train: {y_train.mean():.2%}; test: {y_test.mean():.2%}")


## 4. Разведочный анализ и проверка гипотез


In [ ]:
print("Пропуски, top-10:")
display(df_clean.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))

corr_series = (
    df_clean[numerical_features + ["Fraud_Label"]]
    .corr(numeric_only=True)["Fraud_Label"]
    .drop("Fraud_Label")
    .abs()
    .sort_values(ascending=False)
)
print("Наибольшие абсолютные корреляции с Fraud_Label:")
display(corr_series.head(12).to_frame("|correlation|"))

for feature in [
    "Transaction_Type",
    "Is_International_Transaction",
    "Unusual_Time_Transaction",
    "Is_New_Merchant",
]:
    print(f"Fraud rate by {feature}:")
    display(
        df_clean.groupby(feature, dropna=False)["Fraud_Label"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .rename(columns={"mean": "fraud_rate"})
    )


## 5. Пайплайны и кросс-валидация

Препроцессоры находятся **внутри** `Pipeline`. Поэтому заполнение пропусков,
масштабирование и кодирование категорий каждый раз обучаются только на
тренировочной части соответствующего фолда.


In [ ]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numerical_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            numerical_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "ordinal",
                    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                ),
            ]),
            categorical_features,
        ),
    ]
)

def build_pipeline(model_name, params):
    params = dict(params)
    if model_name == "logreg":
        model = LogisticRegression(
            **params,
            max_iter=1_200,
            random_state=RANDOM_STATE,
        )
        preprocessor = clone(linear_preprocessor)
    elif model_name == "random_forest":
        model = RandomForestClassifier(
            **params,
            n_jobs=1,
            random_state=RANDOM_STATE,
        )
        preprocessor = clone(tree_preprocessor)
    elif model_name == "hist_gb":
        model = HistGradientBoostingClassifier(
            **params,
            random_state=RANDOM_STATE,
        )
        preprocessor = clone(tree_preprocessor)
    else:
        raise ValueError(f"Неизвестная модель: {model_name}")
    return Pipeline([("preprocess", preprocessor), ("classifier", model)])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


In [ ]:
def make_objective(model_name):
    def objective(trial):
        if model_name == "logreg":
            params = {
                "C": trial.suggest_float("C", 1e-3, 30.0, log=True),
                "solver": trial.suggest_categorical("solver", ["lbfgs", "liblinear"]),
                "class_weight": trial.suggest_categorical(
                    "class_weight", [None, "balanced"]
                ),
            }
        elif model_name == "random_forest":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 150, 600),
                "max_depth": trial.suggest_int("max_depth", 6, 30),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_categorical(
                    "max_features", ["sqrt", "log2", None]
                ),
                "class_weight": trial.suggest_categorical(
                    "class_weight", [None, "balanced", "balanced_subsample"]
                ),
            }
        elif model_name == "hist_gb":
            params = {
                "learning_rate": trial.suggest_float(
                    "learning_rate", 0.01, 0.3, log=True
                ),
                "max_iter": trial.suggest_int("max_iter", 150, 600),
                "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 80),
                "max_depth": trial.suggest_int("max_depth", 3, 16),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 200),
                "l2_regularization": trial.suggest_float(
                    "l2_regularization", 1e-8, 10.0, log=True
                ),
                "class_weight": trial.suggest_categorical(
                    "class_weight", [None, "balanced"]
                ),
            }
        else:
            raise ValueError(model_name)

        model = build_pipeline(model_name, params)
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            scoring="roc_auc",
            cv=cv,
            n_jobs=N_JOBS,
        )
        return float(scores.mean())

    return objective


## 6. Подбор гиперпараметров через Optuna


In [ ]:
model_names = ["logreg", "random_forest", "hist_gb"]
studies = {}
best_params = {}

for model_name in model_names:
    print(f"Оптимизация {model_name}...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        study_name=model_name,
    )
    study.optimize(
        make_objective(model_name),
        n_trials=N_TRIALS,
        show_progress_bar=True,
        gc_after_trial=True,
    )
    studies[model_name] = study
    best_params[model_name] = study.best_params
    print(f"Best CV ROC-AUC: {study.best_value:.5f}")
    print(f"Best params: {study.best_params}\n")


## 7. Финальное сравнение на отложенной выборке


In [ ]:
def evaluate_model(model, model_name):
    started = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - started
    prediction = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    report = classification_report(
        y_test,
        prediction,
        output_dict=True,
        zero_division=0,
    )
    return {
        "model": model_name,
        "roc_auc": roc_auc_score(y_test, probability),
        "precision_fraud": report["1"]["precision"],
        "recall_fraud": report["1"]["recall"],
        "f1_fraud": report["1"]["f1-score"],
        "train_time_sec": round(train_time, 2),
    }

display_names = {
    "logreg": "Logistic Regression",
    "random_forest": "Random Forest",
    "hist_gb": "HistGradientBoosting",
}
final_models = {
    name: build_pipeline(name, best_params[name]) for name in model_names
}
results = [
    evaluate_model(final_models[name], display_names[name]) for name in model_names
]
results_df = (
    pd.DataFrame(results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)
display(results_df.round(5))


## 8. Ансамбли: stacking и soft voting


In [ ]:
base_estimators = [
    ("logreg", build_pipeline("logreg", best_params["logreg"])),
    ("rf", build_pipeline("random_forest", best_params["random_forest"])),
    ("hgb", build_pipeline("hist_gb", best_params["hist_gb"])),
]

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(
        C=1.0,
        max_iter=500,
        random_state=RANDOM_STATE,
    ),
    cv=3,
    n_jobs=N_JOBS,
    passthrough=False,
)
voting = VotingClassifier(
    estimators=[
        ("logreg", build_pipeline("logreg", best_params["logreg"])),
        ("rf", build_pipeline("random_forest", best_params["random_forest"])),
        ("hgb", build_pipeline("hist_gb", best_params["hist_gb"])),
    ],
    voting="soft",
    n_jobs=N_JOBS,
)

ensemble_results = [
    evaluate_model(stack, "Stacking Ensemble"),
    evaluate_model(voting, "Soft Voting"),
]
full_results = (
    pd.concat([results_df, pd.DataFrame(ensemble_results)], ignore_index=True)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)
display(full_results.round(5))

best = full_results.iloc[0]
print(
    f"Лучшая модель: {best['model']} | ROC-AUC: {best['roc_auc']:.5f} | "
    f"F1(Fraud): {best['f1_fraud']:.4f}"
)


## 9. Интерпретация и диагностические графики


In [ ]:
rf_pipeline = final_models["random_forest"]
feature_names = rf_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = pd.Series(
    rf_pipeline.named_steps["classifier"].feature_importances_,
    index=feature_names,
)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
top15[::-1].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top-15 Feature Importances — Random Forest")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()
display(top15.to_frame("importance"))


In [ ]:
model_probas = {
    "Logistic Regression": final_models["logreg"].predict_proba(X_test)[:, 1],
    "Random Forest": final_models["random_forest"].predict_proba(X_test)[:, 1],
    "HistGradientBoosting": final_models["hist_gb"].predict_proba(X_test)[:, 1],
    "Stacking": stack.predict_proba(X_test)[:, 1],
    "Voting (soft)": voting.predict_proba(X_test)[:, 1],
}

fig, ax = plt.subplots(figsize=(7, 6))
for name, probability in model_probas.items():
    fpr, tpr, _ = roc_curve(y_test, probability)
    auc = roc_auc_score(y_test, probability)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
fraud_by_prev_count = (
    df_clean.groupby("Previous_Fraud_Count")["Fraud_Label"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "Fraud_Rate", "count": "Transaction_Count"})
)
display(fraud_by_prev_count)

plt.figure(figsize=(10, 5))
sns.barplot(
    x="Previous_Fraud_Count",
    y="Fraud_Rate",
    data=fraud_by_prev_count,
    color="steelblue",
)
plt.title("Impact of Previous Fraud History on Current Fraud Probability")
plt.ylabel("Fraud Rate")
plt.xlabel("Previous Fraud Count")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## Вывод

Проект сравнивает линейную модель, два древовидных алгоритма и ансамбли на
едином разбиении данных. Итоговый выбор следует делать по ROC-AUC вместе с
Recall и Precision мошеннического класса: в прикладном антифроде цена
пропущенной атаки и цена ложной блокировки различаются.
